# OSRM Store to Polygon Distance Calculator
Calculates road distances from each store to its polygon edge points using the free OSRM public API.

**Steps:**
1. Upload your CSV when prompted
2. Wait ~5-10 minutes for processing
3. Download the result CSV with road distances

In [ ]:
# Step 1: Upload your CSV file
from google.colab import files
uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
print(f"\nUploaded: {INPUT_FILE}")

In [ ]:
# Step 2: Load data
import pandas as pd
import requests
import time
from tqdm.notebook import tqdm

df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df):,} rows from {df['store code'].str.strip().nunique()} stores")
df.head()

In [ ]:
# Step 3: Define distance calculation functions

OSRM_BASE = "https://router.project-osrm.org"
BATCH_SIZE = 50
DELAY = 1.5
RETRY_COUNT = 3
TIMEOUT = 30


def get_batch_distances(store_lat, store_lon, point_coords):
    """Use OSRM Table API: one store -> many polygon points in one request."""
    coords = f"{store_lon},{store_lat}"
    for plat, plon in point_coords:
        coords += f";{plon},{plat}"

    destinations = ";".join(str(i) for i in range(1, len(point_coords) + 1))
    url = f"{OSRM_BASE}/table/v1/driving/{coords}?sources=0&destinations={destinations}&annotations=distance"

    for attempt in range(RETRY_COUNT):
        try:
            resp = requests.get(url, timeout=TIMEOUT)
            if resp.status_code == 429:
                time.sleep(5 * (attempt + 1))
                continue

            data = resp.json()
            if data.get("code") == "Ok":
                return [round(d / 1000, 2) if d is not None else None for d in data["distances"][0]]
            else:
                return fallback_individual(store_lat, store_lon, point_coords)
        except Exception as e:
            if attempt < RETRY_COUNT - 1:
                time.sleep(2 * (attempt + 1))
            else:
                return [None] * len(point_coords)

    return [None] * len(point_coords)


def fallback_individual(store_lat, store_lon, point_coords):
    """Fallback: individual route queries if table API fails."""
    results = []
    for plat, plon in point_coords:
        url = f"{OSRM_BASE}/route/v1/driving/{store_lon},{store_lat};{plon},{plat}?overview=false"
        try:
            time.sleep(1)
            resp = requests.get(url, timeout=TIMEOUT)
            data = resp.json()
            if data.get("code") == "Ok" and data.get("routes"):
                results.append(round(data["routes"][0]["distance"] / 1000, 2))
            else:
                results.append(None)
        except Exception:
            results.append(None)
    return results


# Test connection
test = requests.get(f"{OSRM_BASE}/route/v1/driving/77.5946,12.9716;77.6200,13.0000?overview=false", timeout=10)
if test.json().get("code") == "Ok":
    print("OSRM server is reachable! Ready to calculate distances.")
else:
    print("WARNING: OSRM server issue.")

In [ ]:
# Step 4: Calculate all road distances (~5-10 minutes)

df['store code'] = df['store code'].str.strip()
store_groups = df.groupby('store code')
distances = {}  # polygon_edge -> distance

total_points = len(df)
pbar = tqdm(total=total_points, desc="Calculating distances")

for store_name, group in store_groups:
    store_lat = group['Store Latitude'].iloc[0]
    store_lon = group['Store Longitude'].iloc[0]

    polygon_edges = group['polygon edge'].tolist()
    point_lats = group['Point Latitude'].tolist()
    point_lons = group['Point Longitude'].tolist()

    # Process in batches
    for i in range(0, len(polygon_edges), BATCH_SIZE):
        batch_edges = polygon_edges[i:i + BATCH_SIZE]
        batch_coords = list(zip(point_lats[i:i + BATCH_SIZE], point_lons[i:i + BATCH_SIZE]))

        batch_distances = get_batch_distances(store_lat, store_lon, batch_coords)

        for edge, dist in zip(batch_edges, batch_distances):
            distances[edge] = dist

        pbar.update(len(batch_edges))
        time.sleep(DELAY)

pbar.close()
print(f"\nDone! Calculated {len(distances):,} distances.")

In [ ]:
# Step 5: Save results and download

df['road_distance_km'] = df['polygon edge'].map(distances)

success = df['road_distance_km'].notna().sum()
failed = df['road_distance_km'].isna().sum()
valid = df['road_distance_km'].dropna()

print(f"Results:")
print(f"  Successful: {success:,} ({success/len(df)*100:.1f}%)")
print(f"  Failed:     {failed:,} ({failed/len(df)*100:.1f}%)")
if len(valid) > 0:
    print(f"  Avg distance:  {valid.mean():.1f} km")
    print(f"  Max distance:  {valid.max():.1f} km")
    print(f"  Min distance:  {valid.min():.1f} km")
    print(f"  Median:        {valid.median():.1f} km")

OUTPUT_FILE = "store_to_polygon_with_road_distance.csv"
df.to_csv(OUTPUT_FILE, index=False)

# Auto-download
files.download(OUTPUT_FILE)
print(f"\nDownloading {OUTPUT_FILE}...")

In [ ]:
# Optional: Quick stats per store
import matplotlib.pyplot as plt

store_stats = df.groupby('store code')['road_distance_km'].agg(['mean', 'min', 'max']).sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
top20 = store_stats.head(20)
x = range(len(top20))
ax.bar(x, top20['mean'], color='#2196F3', alpha=0.7, label='Avg distance')
ax.errorbar(x, top20['mean'], 
            yerr=[top20['mean'] - top20['min'], top20['max'] - top20['mean']], 
            fmt='none', color='black', capsize=3, label='Min-Max range')
ax.set_xticks(x)
ax.set_xticklabels([s.replace('_wh_hl_01', '') for s in top20.index], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Road Distance (km)')
ax.set_title('Top 20 Stores by Avg Distance to Polygon Edge Points')
ax.legend()
plt.tight_layout()
plt.show()